In [1]:
import os
os.environ.pop("DATABRICKS_CLUSTER_ID", None)
os.environ["DATABRICKS_SERVERLESS_COMPUTE_ID"] = "auto"

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

print(spark.version)
spark.sql("SELECT current_user() AS me, current_catalog() AS catalog").show()
spark.range(1_000_000).selectExpr("sum(id) AS total").show()

4.2.0
+--------------------+-------+
|                  me|catalog|
+--------------------+-------+
|muchaindungu804@g...| sokoni|
+--------------------+-------+

+------------+
|       total|
+------------+
|499999500000|
+------------+



In [2]:
from pyspark.sql import functions as F

n = 1_000_000

sales = (spark.range(n).select(
    F.concat(F.lit("T"), F.lpad(F.col("id").cast("string"), 8, "0")).alias("sale_id"),
    F.concat(F.lit("S"), F.lpad((F.floor(F.rand(1) * 11) + 1).cast("string"), 2, "0")).alias("store_id"),
    F.concat(F.lit("P"), F.lpad((F.floor(F.rand(2) * 32) + 1).cast("string"), 3, "0")).alias("product_id"),
    F.date_add(F.lit("2025-01-01"), F.floor(F.rand(3) * 669).cast("int")).alias("sale_date"),
    (F.floor(F.rand(4) * 5) + 1).cast("int").alias("quantity"),
    F.element_at(F.array(F.lit("M-Pesa"), F.lit("Cash"), F.lit("Card")),
                 (F.floor(F.rand(5) * 3) + 1).cast("int")).alias("payment_method"),
    F.round(F.rand(6) * 600 + 40, 2).alias("unit_price_kes"),
))

(sales.coalesce(8)
      .write.mode("overwrite")
      .option("header", True)
      .csv("abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/sales/"))

In [3]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/sales/"
).count()

1000000

In [4]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/sales/"
).groupBy("store_id").count().orderBy("store_id").show()

+--------+-----+
|store_id|count|
+--------+-----+
|     S01|91075|
|     S02|90882|
|     S03|90986|
|     S04|91309|
|     S05|90768|
|     S06|90370|
|     S07|91339|
|     S08|90469|
|     S09|90704|
|     S10|90767|
|     S11|91331|
+--------+-----+



In [5]:
from pyspark.sql import functions as F

products = spark.createDataFrame([
    ("P001", "Unga Maize Flour 2kg", "Staples", 230),
    ("P002", "Pishori Rice 2kg", "Staples", 310),
    ("P003", "Sugar 1kg", "Staples", 190),
    ("P004", "Cooking Oil 1L", "Staples", 380),
    ("P005", "Wheat Flour 2kg", "Staples", 240),
    ("P006", "Brown Bread 400g", "Bakery", 65),
    ("P007", "Milk 500ml", "Dairy", 60),
    ("P008", "Yoghurt 500ml", "Dairy", 180),
    ("P009", "Butter 250g", "Dairy", 310),
    ("P010", "Eggs Tray 30", "Protein", 480),
    ("P011", "Beef Mince 500g", "Meat", 420),
    ("P012", "Chicken Whole 1kg", "Meat", 690),
    ("P013", "Tilapia 500g", "Fish", 350),
    ("P014", "Tea Leaves 250g", "Beverages", 150),
    ("P015", "Coffee 100g", "Beverages", 260),
    ("P016", "Soda 500ml", "Beverages", 80),
    ("P017", "Juice 1L", "Beverages", 210),
    ("P018", "Mineral Water 1.5L", "Beverages", 120),
    ("P019", "Bar Soap", "Household", 110),
    ("P020", "Washing Powder 1kg", "Household", 340),
    ("P021", "Toothpaste 100g", "Personal Care", 190),
    ("P022", "Sanitary Pads 10s", "Personal Care", 150),
    ("P023", "Tomatoes 1kg", "Fresh Produce", 120),
    ("P024", "Onions 1kg", "Fresh Produce", 110),
    ("P025", "Kale (Sukuma) bunch", "Fresh Produce", 40),
    ("P026", "Bananas (dozen)", "Fresh Produce", 180),
    ("P027", "Potatoes 2kg", "Fresh Produce", 150),
    ("P028", "Biscuits Pack", "Snacks", 70),
    ("P029", "Crisps 100g", "Snacks", 120),
    ("P030", "Maize Cooking Flour 1kg", "Staples", 120),
    ("P031", "Ugali Flour 2kg", "Staples", 200),
    ("P032", "Porridge Flour 500g", "Staples", 90),
], ["product_id", "product_name", "category", "current_price_kes"])

(products.coalesce(1)
         .write.mode("overwrite")
         .option("header", True)
         .csv("abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/products/"))

In [6]:
base = "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/"

sales_count = spark.read.option("header", True).csv(base + "sales/").count()
products_count = spark.read.option("header", True).csv(base + "products/").count()

print("sales:", sales_count)
print("products:", products_count)

sales: 1000000
products: 32


In [7]:
stores = spark.createDataFrame([
    ("S01", "Westlands", "Nairobi", "Nairobi", "Flagship"),
    ("S02", "Eastleigh", "Nairobi", "Nairobi", "Flagship"),
    ("S03", "CBD", "Nairobi", "Nairobi", "Flagship"),
    ("S04", "Mombasa Road", "Nairobi", "Nairobi", "Standard"),
    ("S05", "Mombasa", "Mombasa", "Coast", "Standard"),
    ("S06", "Kisumu", "Kisumu", "Nyanza", "Standard"),
    ("S07", "Nakuru", "Nakuru", "Rift Valley", "Standard"),
    ("S08", "Eldoret", "Eldoret", "Rift Valley", "Standard"),
    ("S09", "Thika", "Thika", "Central", "Small"),
    ("S10", "Machakos", "Machakos", "Eastern", "Small"),
    ("S11", "Nyeri", "Nyeri", "Central", "Small"),
], ["store_id", "store_name", "city", "region", "store_tier"])

(stores.coalesce(1)
       .write.mode("overwrite")
       .option("header", True)
       .csv("abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/stores/"))

In [8]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/stores/"
).count()

11

In [9]:
from pyspark.sql import functions as F

n = 70_000
first_names = ["Wanjiru", "Otieno", "Achieng", "Kamau", "Mwangi", "Njeri", "Kipchoge", "Akinyi", "Mutua", "Wambui"]
last_names = ["Kariuki", "Odhiambo", "Mutiso", "Njoroge", "Kimani", "Chebet", "Ochieng", "Muthoni", "Wekesa", "Gathoni"]
cities = ["Nairobi", "Nairobi", "Nairobi", "Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret", "Thika", "Machakos", "Nyeri"]
regions = ["Nairobi", "Nairobi", "Nairobi", "Nairobi", "Coast", "Nyanza", "Rift Valley", "Rift Valley", "Central", "Eastern", "Central"]

customers = (spark.range(n).select(
    F.concat(F.lit("C"), F.lpad(F.col("id").cast("string"), 6, "0")).alias("customer_id"),
    F.element_at(F.array(*[F.lit(x) for x in first_names]), (F.floor(F.rand(1) * 10) + 1).cast("int")).alias("first_name"),
    F.element_at(F.array(*[F.lit(x) for x in last_names]), (F.floor(F.rand(2) * 10) + 1).cast("int")).alias("last_name"),
    F.col("id").alias("_n"),
))

customers = (customers
    .withColumn("name", F.concat_ws(" ", "first_name", "last_name"))
    .withColumn("email", F.lower(F.concat_ws(".", "first_name", "last_name", F.col("_n").cast("string"))))
    .withColumn("email", F.concat(F.col("email"), F.lit("@example.com")))
    .withColumn("city_idx", (F.floor(F.rand(3) * 11) + 1).cast("int"))
    .withColumn("city", F.element_at(F.array(*[F.lit(x) for x in cities]), F.col("city_idx")))
    .withColumn("region", F.element_at(F.array(*[F.lit(x) for x in regions]), F.col("city_idx")))
    .select("customer_id", "name", "email", "city", "region"))

(customers.coalesce(1)
          .write.mode("overwrite")
          .option("header", True)
          .csv("abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/customers/"))

In [10]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/customers/"
).count()

70000

In [11]:
from pyspark.sql import functions as F

n = 1_000_000

sales = (spark.range(n).select(
    F.concat(F.lit("T"), F.lpad(F.col("id").cast("string"), 8, "0")).alias("sale_id"),
    F.concat(F.lit("C"), F.lpad((F.floor(F.rand(7) * 70000)).cast("string"), 6, "0")).alias("customer_id"),
    F.concat(F.lit("S"), F.lpad((F.floor(F.rand(1) * 11) + 1).cast("string"), 2, "0")).alias("store_id"),
    F.concat(F.lit("P"), F.lpad((F.floor(F.rand(2) * 32) + 1).cast("string"), 3, "0")).alias("product_id"),
    F.date_add(F.lit("2025-01-01"), F.floor(F.rand(3) * 669).cast("int")).alias("sale_date"),
    (F.floor(F.rand(4) * 5) + 1).cast("int").alias("quantity"),
    F.element_at(F.array(F.lit("M-Pesa"), F.lit("Cash"), F.lit("Card")),
                 (F.floor(F.rand(5) * 3) + 1).cast("int")).alias("payment_method"),
    F.round(F.rand(6) * 600 + 40, 2).alias("unit_price_kes"),
))

(sales.coalesce(8)
      .write.mode("overwrite")
      .option("header", True)
      .csv("abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/sales/"))

In [12]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/sales/"
).count()

1000000

In [13]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/sales/"
).select("customer_id").show(5)

+-----------+
|customer_id|
+-----------+
|    C061256|
|    C004290|
|    C006350|
|    C036169|
|    C050867|
+-----------+
only showing top 5 rows


In [14]:
from pyspark.sql import functions as F

cities = ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret", "Thika", "Machakos", "Nyeri"]
regions = ["Nairobi", "Coast", "Nyanza", "Rift Valley", "Rift Valley", "Central", "Eastern", "Central"]

moves = (spark.range(7000).select(
    F.concat(F.lit("C"), F.lpad(F.floor(F.rand(11) * 70000).cast("string"), 6, "0")).alias("customer_id"),
    (F.floor(F.rand(12) * 8) + 1).cast("int").alias("city_idx"),
    F.date_add(F.lit("2025-06-01"), F.floor(F.rand(13) * 514).cast("int")).alias("change_date"),
))

moves = (moves
    .withColumn("city", F.element_at(F.array(*[F.lit(x) for x in cities]), F.col("city_idx")))
    .withColumn("region", F.element_at(F.array(*[F.lit(x) for x in regions]), F.col("city_idx")))
    .select("customer_id", "city", "region", "change_date"))

# About 2% exact duplicate rows, so silver has something to deduplicate
dupes = moves.sample(fraction=0.02, seed=5)
changes = moves.unionByName(dupes)

(changes.coalesce(1)
        .write.mode("overwrite")
        .option("header", True)
        .csv("abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/customer_changes/"))

In [15]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/customer_changes/"
).count()

7130

In [16]:
import random

random.seed(42)

# (item, category, unit, base price in KES for a regular pack)
items = [
    ("Maize Flour", "Staples", "2kg", 230), ("Rice", "Staples", "2kg", 310),
    ("Sugar", "Staples", "1kg", 190), ("Cooking Oil", "Staples", "1L", 380),
    ("Wheat Flour", "Staples", "2kg", 240), ("Ugali Flour", "Staples", "2kg", 200),
    ("Porridge Flour", "Staples", "500g", 90), ("Beans", "Staples", "1kg", 210),
    ("Green Grams", "Staples", "1kg", 230), ("Pasta", "Staples", "500g", 110),
    ("Cornflakes", "Staples", "500g", 420), ("Oats", "Staples", "500g", 260),
    ("Milk", "Dairy", "500ml", 60), ("Yoghurt", "Dairy", "500ml", 180),
    ("Butter", "Dairy", "250g", 310), ("Cheese", "Dairy", "200g", 350),
    ("Cream", "Dairy", "300ml", 240),
    ("Eggs", "Protein", "tray of 30", 480), ("Beef Mince", "Meat", "500g", 420),
    ("Chicken", "Meat", "1kg", 690), ("Tilapia", "Fish", "500g", 350),
    ("Sausages", "Meat", "400g", 290), ("Omena", "Fish", "250g", 120),
    ("Brown Bread", "Bakery", "400g", 65), ("White Bread", "Bakery", "400g", 60),
    ("Mandazi", "Bakery", "pack of 6", 90),
    ("Tea Leaves", "Beverages", "250g", 150), ("Coffee", "Beverages", "100g", 260),
    ("Soda", "Beverages", "500ml", 80), ("Juice", "Beverages", "1L", 210),
    ("Mineral Water", "Beverages", "1.5L", 120), ("Energy Drink", "Beverages", "250ml", 130),
    ("Cocoa Drink", "Beverages", "400g", 320),
    ("Bar Soap", "Household", "150g", 110), ("Washing Powder", "Household", "1kg", 340),
    ("Dishwashing Liquid", "Household", "500ml", 190), ("Toilet Paper", "Household", "4 rolls", 180),
    ("Bleach", "Household", "1L", 150),
    ("Toothpaste", "Personal Care", "100g", 190), ("Shampoo", "Personal Care", "400ml", 380),
    ("Sanitary Pads", "Personal Care", "10 pack", 150), ("Body Lotion", "Personal Care", "400ml", 420),
    ("Tomatoes", "Fresh Produce", "1kg", 120), ("Onions", "Fresh Produce", "1kg", 110),
    ("Kale (Sukuma)", "Fresh Produce", "bunch", 40), ("Bananas", "Fresh Produce", "dozen", 180),
    ("Potatoes", "Fresh Produce", "2kg", 150), ("Avocado", "Fresh Produce", "each", 40),
    ("Cabbage", "Fresh Produce", "each", 80),
    ("Biscuits", "Snacks", "200g", 70), ("Crisps", "Snacks", "100g", 120),
    ("Roasted Groundnuts", "Snacks", "200g", 150),
]

brands = ["Kenblest", "Pwani", "Mama Pima", "Sunrise", "Golden Harvest",
          "Tusker Fresh", "Savanna", "Kibo", "House Brand"]
sizes = [("Small", 0.5), ("Regular", 1.0), ("Family", 2.0)]

# Every item x brand x size combination, shuffled, then take 1,000 of them
combos = [(item, brand, size) for item in items for brand in brands for size in sizes]
random.shuffle(combos)

rows = []
for n, ((name, cat, unit, base), brand, (size, mult)) in enumerate(combos[:1000], start=1):
    price = round(base * mult * random.uniform(0.95, 1.05) / 5) * 5  # round to nearest KES 5
    rows.append((f"P{n:04d}", f"{brand} {name} ({size})", cat, brand, unit, size, int(price)))

products = spark.createDataFrame(rows, [
    "product_id", "product_name", "category", "brand", "unit", "pack_size", "current_price_kes"
])

(products.coalesce(1)
         .write.mode("overwrite")
         .option("header", True)
         .csv("abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/products/"))

In [17]:
p = spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/products/"
)
print("products:", p.count())
p.show(5, truncate=False)

products: 1000
+----------+------------------------------------+-------------+------------+---------+---------+-----------------+
|product_id|product_name                        |category     |brand       |unit     |pack_size|current_price_kes|
+----------+------------------------------------+-------------+------------+---------+---------+-----------------+
|P0001     |House Brand Porridge Flour (Regular)|Staples      |House Brand |500g     |Regular  |90               |
|P0002     |Kibo Bleach (Regular)               |Household    |Kibo        |1L       |Regular  |155              |
|P0003     |Kibo Mandazi (Regular)              |Bakery       |Kibo        |pack of 6|Regular  |90               |
|P0004     |Kenblest Oats (Small)               |Staples      |Kenblest    |500g     |Small    |135              |
|P0005     |Tusker Fresh Sanitary Pads (Family) |Personal Care|Tusker Fresh|10 pack  |Family   |300              |
+----------+------------------------------------+-------------+--

In [18]:
from pyspark.sql import functions as F

base = "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/"

products = (spark.read.option("header", True).csv(base + "products/")
            .select("product_id", F.col("current_price_kes").cast("double").alias("unit_price_kes")))

n = 1_000_000

sales = (spark.range(n).select(
    F.concat(F.lit("T"), F.lpad(F.col("id").cast("string"), 8, "0")).alias("sale_id"),
    F.concat(F.lit("C"), F.lpad(F.floor(F.rand(7) * 70000).cast("string"), 6, "0")).alias("customer_id"),
    F.concat(F.lit("S"), F.lpad((F.floor(F.rand(1) * 11) + 1).cast("string"), 2, "0")).alias("store_id"),
    F.concat(F.lit("P"), F.lpad((F.floor(F.rand(2) * 1000) + 1).cast("string"), 4, "0")).alias("product_id"),
    F.date_add(F.lit("2025-01-01"), F.floor(F.rand(3) * 669).cast("int")).alias("sale_date"),
    (F.floor(F.rand(4) * 5) + 1).cast("int").alias("quantity"),
    F.element_at(F.array(F.lit("M-Pesa"), F.lit("Cash"), F.lit("Card")),
                 (F.floor(F.rand(5) * 3) + 1).cast("int")).alias("payment_method"),
))

sales = sales.join(products, "product_id", "left")

(sales.coalesce(8)
      .write.mode("overwrite")
      .option("header", True)
      .csv(base + "sales/"))

In [19]:
s = spark.read.option("header", True).csv(base + "sales/")
print("rows:", s.count())
print("distinct products:", s.select("product_id").distinct().count())
print("rows with no price:", s.filter("unit_price_kes IS NULL").count())

rows: 1000000
distinct products: 1000
rows with no price: 0


In [20]:
from pyspark.sql import functions as F

base = "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/"

stores = spark.range(11).select(
    F.concat(F.lit("S"), F.lpad((F.col("id") + 1).cast("string"), 2, "0")).alias("store_id"))

prods = spark.range(1000).select(
    F.concat(F.lit("P"), F.lpad((F.col("id") + 1).cast("string"), 4, "0")).alias("product_id"))

weeks = spark.range(96).select(
    F.date_add(F.lit("2025-01-06"), (F.col("id") * 7).cast("int")).alias("week_start"))

inventory = (stores.crossJoin(prods).crossJoin(weeks)
    .select("store_id", "product_id", "week_start",
            F.greatest(F.lit(0), F.floor(F.rand(8) * 60 - 10)).cast("int").alias("quantity_on_hand")))

(inventory.coalesce(8)
          .write.mode("overwrite")
          .option("header", True)
          .csv(base + "inventory/"))

In [21]:
inv = spark.read.option("header", True).csv(base + "inventory/")
print("rows:", inv.count())
print("weeks:", inv.select("week_start").distinct().count())
print("zero-stock share:", inv.filter("quantity_on_hand = 0").count() / inv.count())

rows: 1056000
weeks: 96
zero-stock share: 0.1836098484848485


In [6]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW SCHEMAS IN `Sokoni_retail`").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|samples      |
|sokoni       |
|sokoni_retail|
|system       |
+-------------+

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|silver            |
+------------------+



In [7]:
spark.sql("SELECT current_catalog()").show()
spark.sql("SHOW TABLES IN sokoni_retail.bronze").show(truncate=False)

+-----------------+
|current_catalog()|
+-----------------+
|           sokoni|
+-----------------+

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



In [ ]:
spark.sql("USE CATALOG sokoni_retail")
spark.sql("SELECT current_catalog(), current_schema()").show()


+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|    sokoni_retail|         default|
+-----------------+----------------+



In [13]:
spark.sql("LIST 'abfss://sokoni@sokoniadls.dfs.core.windows.net/'").show(truncate=False)
spark.sql("LIST 'abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/'").show(truncate=False)

+------------------------------------------------------------+-------------+----+-----------------+
|path                                                        |name         |size|modification_time|
+------------------------------------------------------------+-------------+----+-----------------+
|abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/|bronze_layer/|0   |0                |
+------------------------------------------------------------+-------------+----+-----------------+

+----+----+----+-----------------+
|path|name|size|modification_time|
+----+----+----+-----------------+
+----+----+----+-----------------+



In [2]:
spark.sql("SHOW TABLES IN sokoni_retail.bronze").show(truncate=False)

sales = spark.table("sokoni_retail.bronze.sales_raw")
print("rows:", sales.count())
print("duplicate sale_ids:", sales.count() - sales.dropDuplicates(["sale_id"]).count())
print("missing store_id:", sales.filter("store_id IS NULL").count())
sales.printSchema()
sales.show(5, truncate=False)

+--------+------------+-----------+
|database|tableName   |isTemporary|
+--------+------------+-----------+
|bronze  |products_raw|false      |
|bronze  |sales_raw   |false      |
|bronze  |stores_raw  |false      |
+--------+------------+-----------+

rows: 79938
duplicate sale_ids: 791
missing store_id: 1599
root
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- unit_price_kes: double (nullable = true)
 |-- sale_id: string (nullable = true)

+--------+----------+----------+--------+--------------+--------------+--------+
|store_id|product_id|sale_date |quantity|payment_method|unit_price_kes|sale_id |
+--------+----------+----------+--------+--------------+--------------+--------+
|S02     |P021      |2025-07-26|2       |Cash          |187.69        |T0069945|
|S10     |P003      |2025-07-10|5       |M-Pesa        |188.22    